In [0]:
from pyspark.sql import functions as F

In [0]:
bronze_df = spark.table(
    "medallion_catalog.bronze.bronze_orders"
)

bronze_df.show()
bronze_df.printSchema()

In [0]:
bronze_df.select("customer_name").show()

In [0]:
temp_df = bronze_df.withColumn(
    "customer_name",
    F.trim(F.col("customer_name"))
)

In [0]:
temp_df.select("customer_name").show()

In [0]:
temp_df.select("email").show()

In [0]:
temp_df = temp_df.withColumn(
    "email",
    F.lower(F.trim(F.col("email")))
)

In [0]:
temp_df.select("email").show()

In [0]:
temp_df = temp_df.withColumn(
    "email",
    F.when(
        F.col("email").rlike(
            r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
        ),
        F.col("email")
    ).otherwise(None)
)

In [0]:
temp_df.select("email").show()

In [0]:
temp_df.filter(F.col("email").isNull()).count()

In [0]:
temp_df.select("city").show()

In [0]:
temp_df = temp_df.withColumn("city",
    F.lower(F.trim(F.col("city")))
)


In [0]:
temp_df.select("city").show()

In [0]:
temp_df.select("amount").show()

In [0]:
temp_df = temp_df.withColumn(
    "amount",
    F.regexp_replace(
        F.col("amount"),
        "[, ]",
        ""
    )
)

In [0]:
temp_df.select("amount").show()

In [0]:
temp_df = temp_df.withColumn(
    "amount",
    F.when(
        F.trim(F.col("amount")).isin("N/A", "0"),
        None
    ).otherwise(F.col("amount"))
)

In [0]:
temp_df.select("amount").show()

In [0]:
temp_df = temp_df.withColumn(
    "amount",
    F.col("amount").cast("decimal(12,2)")
)

In [0]:
temp_df.select("order_id").show()

In [0]:
temp_df = temp_df.withColumn(
    "order_id",
    F.when(
        F.col("order_id").isNull(),
        "0000"
    ).otherwise(F.col("order_id"))
)

In [0]:
temp_df.select("order_id").show()

In [0]:
temp_df = temp_df.dropDuplicates(["order_id"])

In [0]:
temp_df = temp_df.filter(
    F.col("order_id") != "0000"
)

In [0]:
temp_df = temp_df.withColumn(
    "order_id",
    F.col("order_id").cast("int")
)

In [0]:
temp_df.select("order_date").show()

In [0]:
temp_df = temp_df.withColumn(
    "order_date",
    F.coalesce(
        F.try_to_date(F.col("order_date"), "yyyy-MM-dd"),
        F.try_to_date(F.col("order_date"), "dd/MM/yyyy"),
        F.try_to_date(F.col("order_date"), "yyyy/MM/dd"),
        F.try_to_date(F.col("order_date"), "MM-dd-yyyy"),
        F.try_to_date(F.col("order_date"), "dd-MM-yyyy")
    )
)

In [0]:
temp_df.select("order_date")

In [0]:
temp_df = temp_df.withColumn(
    "status",
    F.initcap(
        F.trim(F.col("status"))
    )
)

In [0]:
temp_df = temp_df.withColumn(
    "amount_valid",
    F.when(
        F.col("amount") >= 0,
        True
    ).otherwise(False)
)

In [0]:
temp_df = temp_df.withColumn(
    "record_valid",
    F.when(
        (F.col("order_id").isNotNull()) &
        (F.col("order_date").isNotNull()) &
        (F.col("amount").isNotNull()) &
        (F.col("amount_valid") == True),
        True
    ).otherwise(False)
)

In [0]:
silver_valid_df = temp_df.filter(
    F.col("record_valid") == True
)



In [0]:
silver_valid_df = silver_valid_df.drop(
    "load_date",
    "source_path",
    "source_file"
)

In [0]:
silver_valid_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("medallion_catalog.silver.silver_orders")

In [0]:
%sql
SELECT * FROM medallion_catalog.silver.silver_orders